# 02 · Baseline classification models

**输入**: `../data/train.csv`

**输出**:
- `../outputs/figures/02_roc_curves.png`
- `../outputs/figures/02_pr_curves.png`
- `../outputs/figures/02_confusion_matrices.png`
- `../outputs/tables/02_baseline_metrics.csv`

**任务**: T6 baseline classifiers
- Logistic Regression (class_weight='balanced')
- Gaussian Naive Bayes
- Decision Tree (max_depth=5, class_weight='balanced')
- Random Forest (n_estimators=300, class_weight='balanced')

Each model is evaluated with 5-fold StratifiedKFold (random_state=42). We use a single 80/20 split for the ROC/PR curve plots so all four lines come from the same validation set.

LightGBM lives in `03_lightgbm_model.ipynb` because it pulls in the ablation experiments and threshold search.

In [ ]:
import os, sys, warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import confusion_matrix

sys.path.insert(0, str(Path('..').resolve() / 'src'))
import data_utils
import evaluation
import features
import models

warnings.filterwarnings('ignore', category=UserWarning)

FIG_DIR = Path('../outputs/figures'); FIG_DIR.mkdir(parents=True, exist_ok=True)
TBL_DIR = Path('../outputs/tables');  TBL_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.dpi'] = 110
RANDOM_STATE = 42

## 1. Load data and split

In [ ]:
train, _ = data_utils.load_data()
feat_cols = data_utils.feature_columns(train)
print('train shape:', train.shape)
print('positive rate:', float(train['target'].mean()))
print('# features:', len(feat_cols))

In [ ]:
# Single 80/20 split powers the ROC / PR / confusion-matrix plots.
X_train, X_valid, y_train, y_valid = data_utils.split_data(train)
X_train_s, X_valid_s, _, _ = features.standardize(X_train, X_valid)
print(X_train.shape, X_valid.shape, 'pos rate train', y_train.mean(), 'valid', y_valid.mean())

## 2. Cross-validated ROC-AUC (5-fold StratifiedKFold)

Reported in the project plan as the headline number. Tree-based models use raw features; LR / NB use standardised features.

In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

X_full, y_full = data_utils.split_xy(train)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

cv_pipelines = {
    'LogisticRegression': Pipeline([('sc', StandardScaler()), ('clf', models.make_logistic_regression())]),
    'GaussianNB':         Pipeline([('sc', StandardScaler()), ('clf', models.make_gaussian_nb())]),
    'DecisionTree':       models.make_decision_tree(),
    'RandomForest':       models.make_random_forest(),
}

cv_records = []
for name, est in cv_pipelines.items():
    scores = cross_val_score(est, X_full, y_full, scoring='roc_auc', cv=skf, n_jobs=-1)
    cv_records.append({'model': name, 'mean_auc': scores.mean(), 'std_auc': scores.std(), 'fold_aucs': scores.tolist()})
    print(f'{name:18s} ROC-AUC = {scores.mean():.4f} ± {scores.std():.4f}  (folds {np.round(scores, 4).tolist()})')

cv_df = pd.DataFrame(cv_records)
cv_df.to_csv(TBL_DIR / '02_cv_roc_auc.csv', index=False)
cv_df

## 3. Hold-out evaluation: full metric bundle

ROC-AUC, PR-AUC, F1, recall, precision, confusion matrix on the 80/20 split.

In [ ]:
fitted = {}
probs = {}
metrics = {}

# (model_factory, uses_scaled_features)
model_specs = [
    ('LogisticRegression', models.make_logistic_regression(), True),
    ('GaussianNB',         models.make_gaussian_nb(),         True),
    ('DecisionTree',       models.make_decision_tree(),       False),
    ('RandomForest',       models.make_random_forest(),       False),
]

for name, est, use_scaled in model_specs:
    Xt = X_train_s if use_scaled else X_train.values
    Xv = X_valid_s if use_scaled else X_valid.values
    est.fit(Xt, y_train)
    p = models.predict_proba_positive(est, Xv)
    fitted[name] = est
    probs[name] = p
    metrics[name] = evaluation.evaluate_binary(y_valid.values, p)
    print(f"{name}: AUC={metrics[name]['roc_auc']:.4f}  PR-AUC={metrics[name]['pr_auc']:.4f}")

metric_table = evaluation.summarise_models(metrics)
metric_table.to_csv(TBL_DIR / '02_baseline_metrics.csv')
metric_table.round(4)

## 4. Combined ROC curves

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 5.5))
for name, p in probs.items():
    pts = evaluation.roc_curve_points(y_valid.values, p)
    ax.plot(pts['fpr'], pts['tpr'], label=f"{name} (AUC={pts['auc']:.4f})")
ax.plot([0, 1], [0, 1], '--', color='gray', alpha=0.6, label='Random')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('Baseline classifiers — ROC curves (hold-out 20%)')
ax.legend(loc='lower right')
fig.tight_layout()
fig.savefig(FIG_DIR / '02_roc_curves.png', dpi=150)
plt.show()

## 5. Combined PR curves

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 5.5))
for name, p in probs.items():
    pts = evaluation.pr_curve_points(y_valid.values, p)
    ax.plot(pts['recall'], pts['precision'], label=f"{name} (AP={pts['ap']:.4f})")
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Baseline classifiers — Precision-Recall curves (hold-out 20%)')
ax.legend(loc='upper right')
fig.tight_layout()
fig.savefig(FIG_DIR / '02_pr_curves.png', dpi=150)
plt.show()

## 6. Confusion matrices (default threshold = 0.5)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4.4))
for ax, (name, p) in zip(axes, probs.items()):
    y_pred = (p >= 0.5).astype(int)
    cm = confusion_matrix(y_valid, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=ax,
                xticklabels=['pred 0', 'pred 1'], yticklabels=['true 0', 'true 1'])
    ax.set_title(f"{name}\nAUC={metrics[name]['roc_auc']:.3f}")
fig.suptitle('Baseline classifiers — confusion matrices (threshold=0.5)', y=1.02)
fig.tight_layout()
fig.savefig(FIG_DIR / '02_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Take-aways

- Logistic Regression and Gaussian NB are strong linear/probabilistic baselines on Santander; the data-generating process is close to the NB independence assumption, so GaussianNB is competitive without any tuning.
- Decision Tree (depth 5) and Random Forest underperform the linear baselines because the signal-per-feature is weak and trees waste splits on noise — see report §5.
- All numbers feed **报告表 1** in the final report. LightGBM and the ablation table are produced in `03_lightgbm_model.ipynb`.